# Project #2: Raster/Imagery Analysis with Google Earth Engine + Database Integration

## Research Question
This notebook explores how summer climate conditions vary across the Kalamazoo area and how those conditions relate to parks and waterways stored through a local DuckDB workflow.

## Objective
This project integrates:
1. Two raster layers from Google Earth Engine
2. Two vector layers managed through DuckDB

### Earth Engine layers
- GRIDMET precipitation (`pr`)
- GRIDMET maximum temperature (`tmmx`)

### Database/vector layers
- OpenStreetMap parks
- OpenStreetMap waterways

## Why this matters
Climate rasters provide environmental context, but they become much more interpretable when compared with landscape features such as parks and waterways. This notebook demonstrates a reproducible geospatial workflow using Earth Engine, geemap, GeoPandas, OSMnx, and DuckDB.


## Package setup

If any required package is missing, uncomment and run the installation cell below once. After installation, restart the kernel and continue.


## Import libraries


In [1]:
import os
import ee
import geemap
import geopandas as gpd
import pandas as pd
import duckdb
import osmnx as ox
from shapely.geometry import box


## Earth Engine authentication and initialization

Google Earth Engine must be authenticated before use. On a machine where authentication has not been completed yet, run `ee.Authenticate()` once. After that, initialize Earth Engine with your Google Cloud project ID.


In [2]:
import ee
ee.Authenticate()

True

In [3]:
ee.Initialize()

In [4]:
ee.Initialize(project="ee-tangihamajumder90")

## Define the study area

A simple bounding box is used for the Kalamazoo area so the workflow stays transparent and reproducible.


In [5]:
xmin, ymin, xmax, ymax = -85.80, 42.15, -85.45, 42.40
aoi_geom = box(xmin, ymin, xmax, ymax)

aoi_gdf = gpd.GeoDataFrame(
    {"name": ["Kalamazoo_AOI"]},
    geometry=[aoi_geom],
    crs="EPSG:4326"
)

aoi_ee = geemap.gdf_to_ee(aoi_gdf)
aoi_ee_geom = aoi_ee.geometry()

aoi_gdf


,name,geometry
0,Kalamazoo_AOI,"POLYGON ((-85.45 42.15, -85.45 42.4, -85.8 42...."


## Select Earth Engine raster datasets

This notebook uses the GRIDMET dataset for summer 2024 climate conditions:
- `pr` for total precipitation
- `tmmx` for mean daily maximum temperature


In [6]:
start_date = "2024-06-01"
end_date = "2024-08-31"

gridmet = (
    ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
    .filterDate(start_date, end_date)
    .filterBounds(aoi_ee_geom)
)

precip = gridmet.select("pr").sum().clip(aoi_ee_geom)
tmax = gridmet.select("tmmx").mean().clip(aoi_ee_geom)


## Visualization parameters

The raster layers are styled for exploratory interpretation. Precipitation is displayed as a blue gradient, while maximum temperature is displayed with a warm color ramp.


In [7]:
precip_vis = {
    "min": 200,
    "max": 450,
    "palette": ["white", "lightblue", "blue", "darkblue"]
}

tmax_vis = {
    "min": 298,
    "max": 304,
    "palette": ["yellow", "orange", "red", "darkred"]
}


## Raster visualization

Because GRIDMET is a coarse-resolution climate raster, the layers appear as gridded cells rather than fine imagery. The two variables are first visualized separately and then compared in a combined interactive map.


In [8]:
Map1 = geemap.Map()
Map1.centerObject(aoi_ee_geom, 10)
Map1.addLayer(precip, precip_vis, "Summer 2024 Precipitation")
Map1.add_gdf(aoi_gdf, layer_name="Study Area")
Map1


Map(center=[42.27505037923222, -85.62499999999902], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
Map2 = geemap.Map()
Map2.centerObject(aoi_ee_geom, 10)
Map2.addLayer(tmax, tmax_vis, "Summer 2024 Max Temperature")
Map2.add_gdf(aoi_gdf, layer_name="Study Area")
Map2


Map(center=[42.27505037923222, -85.62499999999902], controls=(WidgetControl(options=['position', 'transparent_…

## Retrieve vector layers from OpenStreetMap

To satisfy the database component of the project, two vector datasets are retrieved:
1. Parks
2. Waterways

These are downloaded as GeoDataFrames and then stored in DuckDB for analytical querying.


In [10]:
place_name = "Kalamazoo, Michigan, USA"

parks = ox.features_from_place(
    place_name,
    tags={"leisure": "park"}
)

waterways = ox.features_from_place(
    place_name,
    tags={"waterway": True}
)

print("Parks:", len(parks))
print("Waterways:", len(waterways))


Parks: 66
Waterways: 63


## Clean and clip vector layers

The OSM features are standardized to EPSG:4326, invalid geometries are removed, and the layers are clipped to the study area extent.


In [11]:
parks = parks.to_crs("EPSG:4326")
waterways = waterways.to_crs("EPSG:4326")

parks = parks[parks.geometry.notnull()].copy()
waterways = waterways[waterways.geometry.notnull()].copy()

parks = parks.cx[xmin:xmax, ymin:ymax].copy()
waterways = waterways.cx[xmin:xmax, ymin:ymax].copy()

print("Parks after clip:", len(parks))
print("Waterways after clip:", len(waterways))


Parks after clip: 66
Waterways after clip: 63


## Create DuckDB database and load vector attributes

DuckDB is used here as a lightweight analytical database. Geometry is preserved in GeoPandas for mapping, while a WKT version is stored in the database so the imported tables remain queryable and portable.


In [12]:
con = duckdb.connect("project2_v2.duckdb")
parks_db = parks.copy()
waterways_db = waterways.copy()

parks_db["geometry_wkt"] = parks_db.geometry.to_wkt()
waterways_db["geometry_wkt"] = waterways_db.geometry.to_wkt()

con.register("parks_df", parks_db.drop(columns="geometry"))
con.register("waterways_df", waterways_db.drop(columns="geometry"))

con.execute("CREATE OR REPLACE TABLE parks AS SELECT * FROM parks_df")
con.execute("CREATE OR REPLACE TABLE waterways AS SELECT * FROM waterways_df")

print("DuckDB tables created.")


DuckDB tables created.


## Verify database location and contents

The DuckDB file is saved in the notebook's current working directory. The following cells show the save location and confirm the database tables.


In [13]:
print("Current working directory:", os.getcwd())
print("DuckDB file path:", os.path.join(os.getcwd(), "project2.duckdb"))
con.execute("SHOW TABLES").fetchdf()


Current working directory: d:\WesternMichiganUni\GIS_Programming\Project2
DuckDB file path: d:\WesternMichiganUni\GIS_Programming\Project2\project2.duckdb


,name
0,parks
1,parks_df
2,waterways
3,waterways_df


## Basic database exploration


In [14]:
parks_preview = con.execute("SELECT * FROM parks LIMIT 5").df()
waterways_preview = con.execute("SELECT * FROM waterways LIMIT 5").df()

parks_preview


,addr:city,addr:housenumber,addr:postcode,addr:state,addr:street,gnis:feature_id,leisure,name,note,opening_hours,...,ele,access,operator,description,operator:type,name:etymology:wikidata,source,type,operator:wikidata,geometry_wkt
0,None,None,None,None,None,625556,park,Emerald Drive Park,None,None,...,260,yes,City of Kalamazoo,None,public,None,None,multipolygon,Q167155,"POLYGON ((-85.550401 42.25058, -85.550412 42.2..."
1,None,None,None,None,None,None,park,None,None,None,...,None,None,None,None,None,None,None,multipolygon,None,"POLYGON ((-85.589362 42.271052, -85.589298 42...."
2,None,None,None,None,None,None,park,Woods Lake Park,None,None,...,None,None,None,None,None,None,None,multipolygon,None,"POLYGON ((-85.613894 42.261634, -85.613418 42...."
3,Kalamazoo,200,49007,MI,South Rose Street,2627742,park,Bronson Park,Bicycling in the Park is Prohibited,08:00-sunset,...,None,None,None,None,None,None,None,None,None,"POLYGON ((-85.587155 42.290572, -85.587124 42...."
4,None,None,None,None,None,1615557;624665,park,VerSluis & Dickinson Park,None,None,...,237,None,None,None,None,None,None,None,None,"POLYGON ((-85.596308 42.307074, -85.599776 42...."


In [15]:
waterways_preview

,access,canoe,description,fee,informal,surface,tidal,waterway,wheelchair,name,wikidata,intermittent,tunnel,layer,geometry_wkt
0,yes,put_in;egress,Gravel beach leading to trail and parking.,no,no,gravel,no,access_point,no,None,None,None,None,None,POINT (-85.562309 42.291454)
1,None,None,None,None,None,None,None,river,None,Kalamazoo River,None,None,None,None,"LINESTRING (-85.492677 42.28273, -85.493961 42..."
2,None,None,None,None,None,None,None,river,None,Portage Creek,Q35265840,None,None,None,"LINESTRING (-85.573925 42.264552, -85.574387 4..."
3,None,None,None,None,None,None,None,stream,None,Arcadia Creek,None,None,culvert,None,"LINESTRING (-85.588734 42.292958, -85.588372 4..."
4,None,None,None,None,None,None,None,stream,None,Arcadia Creek,None,None,None,None,"LINESTRING (-85.589636 42.292951, -85.588734 4..."


In [16]:
parks_count = con.execute("SELECT COUNT(*) AS n_parks FROM parks").df()
waterways_count = con.execute("SELECT COUNT(*) AS n_waterways FROM waterways").df()

display(parks_count)
display(waterways_count)


,n_parks
0,66


,n_waterways
0,63


## Compute geometric summaries in GeoPandas

Areas and lengths are calculated in a projected coordinate reference system so that the measurements are in meters.


In [17]:
parks_proj = parks.to_crs("EPSG:26985")
waterways_proj = waterways.to_crs("EPSG:26985")

parks_proj["park_area_m2"] = parks_proj.geometry.area
waterways_proj["waterway_length_m"] = waterways_proj.geometry.length

print("Total park area (sq km):", parks_proj["park_area_m2"].sum() / 1_000_000)
print("Total waterway length (km):", waterways_proj["waterway_length_m"].sum() / 1000)


Total park area (sq km): 3.9747765164641047
Total waterway length (km): 46.51152389493974


## Prepare vector layers for Earth Engine zonal statistics

The park and waterway layers are converted to Earth Engine FeatureCollections. A small buffer is applied to waterways so that area-based raster summaries are more stable.


In [18]:
waterways_buffered = waterways_proj.copy()
waterways_buffered["geometry"] = waterways_buffered.geometry.buffer(100)
waterways_buffered = waterways_buffered.to_crs("EPSG:4326")

parks_ee = geemap.gdf_to_ee(parks)
waterways_ee = geemap.gdf_to_ee(waterways_buffered)

print("Converted GeoDataFrames to Earth Engine FeatureCollections.")


Converted GeoDataFrames to Earth Engine FeatureCollections.


## Zonal statistics

Raster summaries are computed over parks and buffered waterways using Earth Engine.


In [19]:
parks_precip_stats = precip.reduceRegions(
    collection=parks_ee,
    reducer=ee.Reducer.mean(),
    scale=4000
)

parks_tmax_stats = tmax.reduceRegions(
    collection=parks_ee,
    reducer=ee.Reducer.mean(),
    scale=4000
)

waterways_precip_stats = precip.reduceRegions(
    collection=waterways_ee,
    reducer=ee.Reducer.mean(),
    scale=4000
)

waterways_tmax_stats = tmax.reduceRegions(
    collection=waterways_ee,
    reducer=ee.Reducer.mean(),
    scale=4000
)


In [20]:
parks_precip_df = geemap.ee_to_df(parks_precip_stats)
parks_tmax_df = geemap.ee_to_df(parks_tmax_stats)

waterways_precip_df = geemap.ee_to_df(waterways_precip_stats)
waterways_tmax_df = geemap.ee_to_df(waterways_tmax_stats)

print("Park precip stats rows:", len(parks_precip_df))
print("Park tmax stats rows:", len(parks_tmax_df))
print("Waterway precip stats rows:", len(waterways_precip_df))
print("Waterway tmax stats rows:", len(waterways_tmax_df))


Park precip stats rows: 66
Park tmax stats rows: 66
Waterway precip stats rows: 63
Waterway tmax stats rows: 63


In [21]:
if "mean" in parks_precip_df.columns:
    parks_precip_df = parks_precip_df.rename(columns={"mean": "park_precip_mean"})
if "mean" in parks_tmax_df.columns:
    parks_tmax_df = parks_tmax_df.rename(columns={"mean": "park_tmax_mean"})
if "mean" in waterways_precip_df.columns:
    waterways_precip_df = waterways_precip_df.rename(columns={"mean": "waterway_precip_mean"})
if "mean" in waterways_tmax_df.columns:
    waterways_tmax_df = waterways_tmax_df.rename(columns={"mean": "waterway_tmax_mean"})

display(parks_precip_df.head())
display(parks_tmax_df.head())


,access,ele,element,gnis:feature_id,id,leisure,park_precip_mean,name,operator,operator:type,...,addr:housenumber,addr:postcode,addr:state,addr:street,note,opening_hours,website,description,name:etymology:wikidata,source
0,yes,260,relation,625556,15328449,park,514.899999,Emerald Drive Park,City of Kalamazoo,public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,relation,NaN,19020918,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,relation,NaN,19166164,park,NaN,Woods Lake Park,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,way,2627742,98341001,park,483.299999,Bronson Park,NaN,NaN,...,200,49007,MI,South Rose Street,Bicycling in the Park is Prohibited,08:00-sunset,https://www.kzooparks.org/Parks-Facilities/Bro...,NaN,NaN,NaN
4,NaN,237,way,1615557;624665,100854680,park,454.899999,VerSluis & Dickinson Park,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,access,ele,element,gnis:feature_id,id,leisure,park_tmax_mean,name,operator,operator:type,...,addr:housenumber,addr:postcode,addr:state,addr:street,note,opening_hours,website,description,name:etymology:wikidata,source
0,yes,260,relation,625556,15328449,park,300.627472,Emerald Drive Park,City of Kalamazoo,public,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,relation,NaN,19020918,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,relation,NaN,19166164,park,NaN,Woods Lake Park,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,way,2627742,98341001,park,300.124176,Bronson Park,NaN,NaN,...,200,49007,MI,South Rose Street,Bicycling in the Park is Prohibited,08:00-sunset,https://www.kzooparks.org/Parks-Facilities/Bro...,NaN,NaN,NaN
4,NaN,237,way,1615557;624665,100854680,park,300.037354,VerSluis & Dickinson Park,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Final interactive map

This combined map overlays the two climate rasters with parks, waterways, and the study area boundary. Opacity is reduced so that the temperature layer does not completely hide the precipitation layer.


In [22]:
Map = geemap.Map()
Map.centerObject(aoi_ee_geom, 10)

Map.addLayer(precip, precip_vis, "Precipitation",opacity=0.75)
Map.addLayer(tmax, tmax_vis, "Temperature")

Map.add_gdf(aoi_gdf, layer_name="Study Area")
Map.add_gdf(parks, layer_name="Parks")
Map.add_gdf(waterways, layer_name="Waterways")

Map


Map(center=[42.27505037923222, -85.62499999999902], controls=(WidgetControl(options=['position', 'transparent_…

## Summary tables

The following tables summarize both the vector database content and the Earth Engine raster statistics derived over those features.


In [23]:
summary_dict = {
    "Total number of parks": len(parks),
    "Total number of waterways": len(waterways),
    "Total park area (sq km)": round(parks_proj["park_area_m2"].sum() / 1_000_000, 2),
    "Total waterway length (km)": round(waterways_proj["waterway_length_m"].sum() / 1000, 2),
}

summary_df = pd.DataFrame(list(summary_dict.items()), columns=["Metric", "Value"])
summary_df


,Metric,Value
0,Total number of parks,66.00
1,Total number of waterways,63.00
2,Total park area (sq km),3.97
3,Total waterway length (km),46.51


This workflow demonstrates how summer precipitation and maximum temperature patterns can be explored in relation to local parks and waterways. Parks represent vegetated and recreational land-use features, while waterways represent hydrologically significant landscape elements. By storing these features in DuckDB and summarizing Earth Engine raster values over them, the notebook connects climate surfaces to real-world environmental features and provides a foundation for further analysis of how climate conditions vary across landscape types.

In [24]:
park_precip_mean_over_features = None
park_tmax_mean_over_features = None
waterway_precip_mean_over_features = None
waterway_tmax_mean_over_features = None

if "park_precip_mean" in parks_precip_df.columns:
    park_precip_mean_over_features = parks_precip_df["park_precip_mean"].dropna().mean()

if "park_tmax_mean" in parks_tmax_df.columns:
    park_tmax_mean_over_features = parks_tmax_df["park_tmax_mean"].dropna().mean()

if "waterway_precip_mean" in waterways_precip_df.columns:
    waterway_precip_mean_over_features = waterways_precip_df["waterway_precip_mean"].dropna().mean()

if "waterway_tmax_mean" in waterways_tmax_df.columns:
    waterway_tmax_mean_over_features = waterways_tmax_df["waterway_tmax_mean"].dropna().mean()

ee_summary = pd.DataFrame({
    "Feature_Type": ["Parks", "Parks", "Waterways", "Waterways"],
    "Statistic": ["Mean Summer Precipitation", "Mean Summer Tmax", "Mean Summer Precipitation", "Mean Summer Tmax"],
    "Value": [
        park_precip_mean_over_features,
        park_tmax_mean_over_features,
        waterway_precip_mean_over_features,
        waterway_tmax_mean_over_features
    ]
})

ee_summary


,Feature_Type,Statistic,Value
0,Parks,Mean Summer Precipitation,479.338889
1,Parks,Mean Summer Tmax,300.171611
2,Waterways,Mean Summer Precipitation,491.487601
3,Waterways,Mean Summer Tmax,300.297992


## Interpretation

The analysis integrates climate raster data from Google Earth Engine with vector features representing parks and waterways to explore how environmental conditions vary across landscape types.

Parks represent vegetated and green spaces, while waterways represent hydrological features such as rivers and streams. By applying zonal statistics, the study summarizes precipitation and temperature values over these features, allowing for a comparison between different environmental settings.

The results indicate that climate variables such as precipitation and maximum temperature can be associated with landscape characteristics. Parks, which are typically covered with vegetation, may experience slightly moderated temperature conditions compared to surrounding areas. Similarly, waterways represent important hydrological components that interact with precipitation patterns.

However, it is important to note that the GRIDMET dataset has a relatively coarse spatial resolution (~4 km), meaning that the results represent generalized climate conditions rather than fine-scale variations. Therefore, the observed patterns should be interpreted as broad environmental trends rather than precise local measurements.

Overall, this workflow demonstrates how raster and vector datasets can be integrated to provide meaningful environmental insights. The approach can be extended in future studies by incorporating higher-resolution datasets or additional variables such as land use or population data to better understand human-environment interactions.


## Conclusion

This notebook demonstrated a complete Earth Engine plus DuckDB workflow. Two climate rasters from GRIDMET were processed in Earth Engine, two vector layers from OpenStreetMap were stored in DuckDB, and zonal statistics were used to connect the raster and vector components. The resulting maps and tables provide an exploratory environmental analysis for the Kalamazoo area and establish a foundation for future work with additional variables, finer-resolution boundaries, or more advanced spatial summaries.


## Optional export

The following cell saves summary outputs and vector layers for later reuse.


In [25]:
summary_df.to_csv("project2_summary_metrics.csv", index=False)
ee_summary.to_csv("project2_ee_feature_summaries.csv", index=False)

parks.to_file("parks.geojson", driver="GeoJSON")
waterways.to_file("waterways.geojson", driver="GeoJSON")

print("Outputs saved.")


Outputs saved.
